# to do:
Two more combinations to explore:
1) subset_frac = 0.001, all_unknown_zero = False
2) subset_frac = 0.01, all_unknown_zero = True

In [103]:
subset_frac = 0.01
all_unknown_zero = False # whether to initialize actual unknowns with 0 as well


In [104]:
import os
import itertools

import pandas as pd
import numpy as np

from tqdm import tqdm

import torch 

from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
import scLEMBAS.utilities as utils
# from scLEMBAS.predict import get_prediction

sys.path.insert(1, '../../.')
from McCauley_utils import all_data, initialize_mod_and_trainer

sys.path.insert(1, '../../../.') 
from notebook_utils import get_split

In [105]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)


In [106]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data

known_moa_mask = sn_ppis[weight_label].isin([1, -1])
known_moa = sn_ppis[known_moa_mask].copy()
sign_frequency = known_moa[weight_label].value_counts(normalize = True)
p, q = sign_frequency[1.0], sign_frequency[-1.0]


sn_ppis_all = sn_ppis.copy()
sn_ppis_all['interaction_'] = sn_ppis_all[[source_label, target_label]].agg('^'.join, axis=1).tolist()  

In [107]:
n_ensembles = 5
seed_multiplier = 21234
def load_model(fold, ensemble_idx, from_trainer = False):
    """Loads the model and training object.

    Two different ways to do so: from pickled training object (larger files) or from model state dict `.pt` file (smaller files to transfer).

    Parameters
    ----------
    fold : int
        fold split
    ensemble_idx : int
        ensemble index
    from_trainer : bool, optional
        whether to load from trainer object or model state dict, by default False
        if False, the training object is not returned
    """
    frac_str = "{:d}".format(int(round(subset_frac * 1000)))  # 3 decimal precision
    fn_label = "{}_fold{}_ensemble{}_frac{}".format(author, fold, ensemble_idx, frac_str)
    if all_unknown_zero:
        fn_label += '_zeroinit'
    fn_base = os.path.join(data_path, 'processed', 'moa_ensembles', fn_label)
    
    if from_trainer:
        fn_trainer =  os.path.join(fn_base + '_moa_trainer_actual.pickle'.format(ensemble_idx))
        trainer = io.read_pickled_object(fn_trainer)
        mod = trainer.mod
    else:
        seed_ = seed + ensemble_idx + 1 + (seed_multiplier * ensemble_idx * fold) if ensemble_idx <= 3 else seed
        mod, trainer = initialize_mod_and_trainer(
            fold = fold, 
            adversarial_penalty = True, 
            randomize = False, 
            seed = seed_
        )
        
        fn_mod = os.path.join(fn_base + '_moa_model_actual.pt'.format(ensemble_idx))

            
        mod.load_state_dict(torch.load(fn_mod))
        trainer = None
        
    moa_removed = open(fn_base + '_moa_removededges.txt').read().splitlines()
    return mod, trainer, moa_removed
        
    



In [108]:
res = []
for (fold, ensemble_idx) in tqdm(itertools.product(range(5), range(n_ensembles))):
    mod, trainer, moa_removed = load_model(fold, ensemble_idx, from_trainer = False)

    actual_moa = sn_ppis_all.set_index('interaction_').loc[moa_removed, weight_label].values
    assert sorted(set(sn_ppis_moa[weight_label])) == [-1, 1], 'Unexpected MOAs missing/present'

    n = len(moa_removed)
    sources = torch.empty(n, device=mod.device, dtype=torch.int32)
    targets = torch.empty(n, device=mod.device, dtype=torch.int32)

    for i, ppi in enumerate(moa_removed):
        source_id, target_id = ppi.split('^')
        sources[i] = mod.node_idx_map[source_id]
        targets[i] = mod.node_idx_map[target_id]

    pred_moa = np.sign(mod.signaling_network.weights.data[(targets, sources)].cpu().numpy())

    res_ = pd.DataFrame({
        'interaction': moa_removed, 
        'actual_moa': actual_moa, 
        'pred_moa': pred_moa, 
    })
    res_['fold'] = fold
    res_['ensemble_idx'] = ensemble_idx
    res.append(res_)
    
res = pd.concat(res, ignore_index = True)

25it [01:16,  3.05s/it]


In [109]:
pred_sign_freq = pd.Series(res.pred_moa).value_counts(normalize = True)

print(f"Class frequencies actual: activating={p:.3f}, inhibiting={q:.3f}")
print(f"Class frequencies predicted: activating={pred_sign_freq[1.0]:.3f}, inhibiting={pred_sign_freq[-1.0]:.3f}")


print()
print('----Random baselines----')
rand_acc_freq = p**2 + q**2
rand_acc_max = max(p,q)

print('Guessing by class frequency: {:2f}'.format(rand_acc_freq))
print('Guessing by most frequent class: {:2f}'.format(rand_acc_max))
print('Guessing uniform random: 0.5')


print()
print('----Prediction metrics----')

acc = np.mean(res.pred_moa == res.actual_moa)
bac = balanced_accuracy_score(res.actual_moa, res.pred_moa)
mcc = matthews_corrcoef(res.actual_moa, res.pred_moa)
kappa = cohen_kappa_score(res.actual_moa, res.pred_moa)

print(f"Accuracy: {acc:.3f}")
print(f"Balanced Accuracy: {bac:.3f}  (random=0.5)")
print(f"MCC:               {mcc:.3f}  (random=0.0)")
print(f"Cohen's Kappa:     {kappa:.3f}  (random=0.0)")

Class frequencies actual: activating=0.715, inhibiting=0.285
Class frequencies predicted: activating=0.671, inhibiting=0.329

----Random baselines----
Guessing by class frequency: 0.592182
Guessing by most frequent class: 0.714688
Guessing uniform random: 0.5

----Prediction metrics----
Accuracy: 0.581
Balanced Accuracy: 0.508  (random=0.5)
MCC:               0.015  (random=0.0)
Cohen's Kappa:     0.015  (random=0.0)


In [110]:
mask_act = res.actual_moa == 1
mask_inh = res.actual_moa == -1
print(f"Accuracy on activating:  {np.mean(res.pred_moa[mask_act] == 1):.3f}")
print(f"Accuracy on inhibiting:  {np.mean(res.pred_moa[mask_inh] == -1):.3f}")

Accuracy on activating:  0.675
Accuracy on inhibiting:  0.341
